# Flow Chart Pengerjaan:
1. Cover Image
2. DCT & Quantization
3. Block Smoothness Estimation & Sorting
4. Optimal Zero Pair AC Selection 
5. Zigzag Scan
6. Zero AC Pair Construction
7. Adaptive Payload Assignment
8. Preditiction Error Expansion Based On Turtle Shell Embedding
9. Stego DCT Coefficients
10. Entropy Coding
11. Stego Image

In [72]:
from PIL import Image
from performance import psnr, fsi, ssim
from zigzag import zigzag, inverse_zigzag
from math import ceil, floor, log2, log10, sqrt
import copy
import cv2
import jpeglib
import numpy as np
import import_ipynb
import matplotlib.pyplot as plt
import turtleShell
import FrequencyDomain as FD

In [73]:
# GLOBAL VARIABLES
Gp = 5
Gn = -5

In [74]:
def get_quantized_coefficients(image_path):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, _, _  = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

In [75]:
def convert_data_to_bits(data):
    data = data + '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in data)
    lendata = len(data_bin)
    return data_bin, lendata

In [76]:
def causal_neighboor_smoothness(image_path):
    coeffs = get_quantized_coefficients(image_path)
    smoothness_block = []
    for idx in range(1, len(coeffs)):
        sum_z_k = np.sum(coeffs[idx-1][1:] == 0)
        sum_ac_k = np.sum(np.abs(coeffs[idx-1][1:]))
        smoothness_block.append(((idx), sum_z_k, sum_ac_k))
    mean_ac = np.mean([block[2] for block in smoothness_block])
    return smoothness_block, int(mean_ac)

In [77]:
def optimal_zero_pair_selection(image_path, threshold, payload=None, s_block=None, t_smooth=None):
    coeffs = get_quantized_coefficients(image_path) 
    num_pairs = (len(coeffs[0]) - 1) // 2
    bits_at_position = [0] * num_pairs 
    zero_pair_counts = [0] * num_pairs 
    t_select = 0
    total_ec_global = 0 

    for idx in range(1, len(coeffs)):
        sum_ac_prev = np.sum(np.abs(coeffs[idx-1][1:]))
        N = 4 if (sum_ac_prev > t_smooth) else 3
        idx_pairs = 0
        for k in range(2, 64, 2): 
            if idx_pairs >= threshold: break
            e1 = coeffs[idx][k-1] - coeffs[idx - 1][k-1]
            e2 = coeffs[idx][k] - coeffs[idx - 1][k]
            if e1 == 0 and e2 == 0:
                bits_at_position[idx_pairs] += N
                zero_pair_counts[idx_pairs] += 1
                total_ec_global += N
            idx_pairs += 1

    if payload:
        if total_ec_global < payload:
            raise ValueError(f"Warning: Kapasitas total ({total_ec_global}) tidak mencukupi payload ({payload})")    
        else:
            current_accumulated_bits = 0
            for i in range(len(bits_at_position)):
                current_accumulated_bits += bits_at_position[i]
                if current_accumulated_bits >= payload:
                    t_select = i + 1 
                    break

    print(f"Statistik Zero Pairs per Posisi Zigzag: {zero_pair_counts}")
    print(f"Kapasitas per posisi zigzag (bits): {bits_at_position}")
    print(f"Optimal Threshold T yang dipilih: {t_select}")
    print(f"Total Kapasitas tersedia: {total_ec_global} bits")
    return t_select, total_ec_global, zero_pair_counts

In [78]:
def construct_stego_file(image_path, new_coeffs):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            im.Y[i, j] = block
            idx += 1

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Image with secret data is saved to {output_path}")
    im.write_dct(output_path)

In [79]:
def recovered_stego_file(image_path, new_coeffs):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            im.Y[i, j] = block
            idx += 1

    output_path = "recovered-images/recovered_" + image_path.split("/")[-1]
    print(f"Recovered image saved to {output_path}")
    im.write_dct(output_path)

In [80]:
def rdh_embed_process(image_path, data_bin, t_select, s_block, t_smooth):
    coeffs = get_quantized_coefficients(image_path) 
    orig_coeffs = copy.deepcopy(coeffs) 
    data_idx = 0
    lendata = len(data_bin)
    
    for idx in range(1, len(coeffs)):
        sum_ac_prev = np.sum(np.abs(orig_coeffs[idx-1][1:]))
        N = 4 if (sum_ac_prev < t_smooth) else 3
        mode = "8N" if N == 3 else "17N"
        radius = 1 if N == 3 else 2 
        
        for k in range(2, t_select * 2 + 1, 2):
            e1 = int(orig_coeffs[idx][k-1] - orig_coeffs[idx - 1][k-1])
            e2 = int(orig_coeffs[idx][k] - orig_coeffs[idx - 1][k])
            e1_mod, e2_mod = e1, e2
            if e1 == 0 and e2 == 0 and data_idx < lendata:
                bits = data_bin[data_idx : data_idx + N].ljust(N, '0')
                data_idx += N
                target_val = int(bits, 2)
                candidate_coords = turtleShell.get_kxk_nearest_zero(0, 0, N)
                e1_mod, e2_mod = turtleShell.find_val_from_zero(candidate_coords, target_val, (e1, e2), mode=mode)
            elif not (e1 == 0 and e2 == 0):
                e1_mod = e1 + int(np.sign(e1)) * radius if e1 != 0 else 0
                e2_mod = e2 + int(np.sign(e2)) * radius if e2 != 0 else 0

            coeffs[idx][k-1] = orig_coeffs[idx - 1][k-1] + e1_mod
            coeffs[idx][k] = orig_coeffs[idx - 1][k] + e2_mod

    construct_stego_file(image_path, coeffs)
    return coeffs

In [81]:
def rdh_extract_process(stego_image_path, t_select, t_smooth):
    coeffs = get_quantized_coefficients(stego_image_path) 
    bit_stream = ""    
    secret_data = "" 
    stop_extraction = False

    for idx in range(1, len(coeffs)):
        sum_ac_k = np.sum(np.abs(coeffs[idx-1][1:]))
        N = 4 if (sum_ac_k < t_smooth) else 3
        mode = "8N" if N == 3 else "17N" 
        radius = 1 if N == 3 else 2 

        for k in range(2, t_select * 2 + 1, 2): 
            d1 = int(coeffs[idx][k-1] - coeffs[idx - 1][k-1])
            d2 = int(coeffs[idx][k] - coeffs[idx - 1][k])
            e1, e2 = d1, d2
            if turtleShell.is_in_central_shell(d1, d2, mode=mode):
                if not stop_extraction:
                    val = turtleShell.get_zero_matrix_value(d1, d2, mode=mode)
                    bit_stream += format(val, f'0{N}b')
                    while len(bit_stream) >= 8:
                        byte = bit_stream[:8]
                        bit_stream = bit_stream[8:]
                        char_val = int(byte, 2)
                        if char_val == 0: 
                            stop_extraction = True
                            break
                        secret_data += chr(char_val)
                e1, e2 = 0, 0
            else:
                e1 = d1 - int(np.sign(d1)) * radius if d1 != 0 else 0
                e2 = d2 - int(np.sign(d2)) * radius if d2 != 0 else 0

            coeffs[idx][k-1] = coeffs[idx - 1][k-1] + e1
            coeffs[idx][k] = coeffs[idx - 1][k] + e2

    recovered_stego_file(stego_image_path, coeffs)
    return secret_data

In [82]:
def read_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    return content

In [83]:
# RDH with Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def rdh_encode(image_path, secret_data):
    ori_coeff = get_quantized_coefficients(image_path)
    data_bin, lendata = convert_data_to_bits(secret_data)
    print(secret_data)
    print(f"Data bits: {data_bin}")
    print(f"Data bits length: {lendata}")    
    s_block, mean_thresold = causal_neighboor_smoothness(image_path)
    t_select, _, _ = optimal_zero_pair_selection(image_path, 31, payload=len(secret_data)*8, s_block=s_block, t_smooth=mean_thresold)
    coeff_after_embed = rdh_embed_process(image_path, data_bin, t_select, s_block, mean_thresold)
    diff = np.array(ori_coeff) - np.array(coeff_after_embed)
    return ori_coeff, coeff_after_embed, diff, t_select, mean_thresold

In [84]:
# RDH with Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def rdh_decode(stego_image_path, t_select = 21, t_smooth=2):
    secret_data = rdh_extract_process(stego_image_path, t_select=t_select, t_smooth=t_smooth)
    return secret_data

In [85]:
pay_size = 80
cover_folder = "cover-images/"
stego_folder = "stego-images/"
payload_folder = "payload/"
cover_image_path = f"baboon_qf90.jpeg"
stego_image_path = f"stego_baboon_qf90.jpeg"
data = read_text_file(f"{payload_folder}{pay_size}Kb.txt")
ori_coeffs, embedded_coeffs, diff, t_select, mean_thresold = rdh_encode(f"{cover_folder}{cover_image_path}", data)

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Vestibulum bibendum hendrerit purus, quis pellentesque ligula placerat et. Donec quam nisi, vehicula a arcu nec, commodo imperdiet lorem. Donec hendrerit leo non odio auctor efficitur. Nullam quis odio eu velit fermentum molestie quis eget ex. Praesent imperdiet, lorem sit amet tempor congue, sem lacus ultrices enim, vel laoreet sapien turpis ut felis. Duis accumsan, massa ac aliquam viverra, orci ante gravida lectus, id molestie augue dolor lacinia velit. Vestibulum porta, purus sit amet mollis vehicula, erat lectus pulvinar lacus, quis venenatis justo purus eget enim. Aliquam erat volutpat. Morbi malesuada tincidunt lectus, et luctus leo. Maecenas augue orci, aliquet at diam vel, maximus volutpat enim. Morbi nec scelerisque lectus. Phasellus sagittis, ex sagittis consequat laoreet, dui dui sodales urna, at congue sem ante at diam. Phasellus sodales orci ac risus hendrerit, ut rutrum nisl aliquet. Vestibulum erat urna, consequat

In [86]:
secret_data = rdh_decode(f"{stego_folder}{stego_image_path}", t_select=t_select, t_smooth=mean_thresold)
print("Extracted Data:", secret_data)

Recovered image saved to recovered-images/recovered_stego_baboon_qf90.jpeg
Extracted Data: Lorem ipsum dolor sit amet, consectetur adipiscing elit. Vestibulum bibendum hendrerit purus, quis pellentesque ligula placerat et. Donec quam nisi, vehicula a arcu nec, commodo imperdiet lorem. Donec hendrerit leo non odio auctor efficitur. Nullam quis odio eu velit fermentum molestie quis eget ex. Praesent imperdiet, lorem sit amet tempor congue, sem lacus ultrices enim, vel laoreet sapien turpis ut felis. Duis accumsan, massa ac aliquam viverra, orci ante gravida lectus, id molestie augue dolor lacinia velit. Vestibulum porta, purus sit amet mollis vehicula, erat lectus pulvinar lacus, quis venenatis justo purus eget enim. Aliquam erat volutpat. Morbi malesuada tincidunt lectus, et luctus leo. Maecenas augue orci, aliquet at diam vel, maximus volutpat enim. Morbi nec scelerisque lectus. Phasellus sagittis, ex sagittis consequat laoreet, dui dui sodales urna, at congue sem ante at diam. Phasell

In [87]:
# Test performance metrics
psnr_value = psnr(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
fsi_value = fsi(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
ssim_value = ssim(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
print(f"PSNR: {psnr_value} dB")
print(f"FSI: {fsi_value}")
print(f"SSIM: {ssim_value}")

Size cover: 226275
Size stego: 233259
PSNR: 45.545318732950754 dB
FSI: 6984.0
SSIM: 0.9948097105816933


In [88]:
# Test performance metrics
recovered_folder = "recovered-images/"
recovered_image = f"recovered_{stego_image_path}"
psnr_value = psnr(f"{cover_folder}{cover_image_path}", f"{recovered_folder}{recovered_image}")
fsi_value = fsi(f"{cover_folder}{cover_image_path}", f"{recovered_folder}{recovered_image}")
ssim_value = ssim(f"{cover_folder}{cover_image_path}", f"{recovered_folder}{recovered_image}")
print(f"PSNR: {psnr_value} dB")
print(f"FSI: {fsi_value}")
print(f"SSIM: {ssim_value}")

Size cover: 226275
Size stego: 226311
PSNR: inf dB
FSI: 36.0
SSIM: 1.0


In [89]:
# DEBUG
print("Difference Coefficients:")
for i in range(len(diff)):
    print(diff[i])

print("-" * 30)

print("Coeff Before Embedded:")
for i in range(len(ori_coeffs)):
    print(ori_coeffs[i])

print("-" * 30)

print("Coeff After Embedded:")
for i in range(len(embedded_coeffs)):
    print(embedded_coeffs[i])

Difference Coefficients:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[ 0.  1. -1. -1.  1. -1.  1.  1.  1. -1.  0.  1. -1.  1. -1.  1. -1.  1.
  1. -1.  1. -1.  1. -1.  1.  1. -1.  1.  0. -1.  1.  0.  1. -1. -1. -1.
 -1.  0. -1.  1.  1.  1.  1. -1. -1.  0. -1. -1. -1.  0.  1.  0.  1.  1.
 -1.  1.  0.  1.  1.  0.  0.  1. -1.  0.]
[ 0. -1. -1.  1. -1. -1.  1. -1. -1.  1.  1. -1. -1. -1.  1. -1.  1. -1.
 -1.  1.  1.  1. -1.  0. -1. -1. -1. -1. -1.  1. -1.  0. -1.  0.  1. -1.
 -1.  0. -1. -1. -1. -1.  1.  1.  1.  1.  0.  1.  1.  0. -1.  0. -1.  0.
  1.  1.  0. -1. -1. -1.  0. -1.  0.  0.]
[ 0.  1.  1.  1.  1.  1.  1. -1.  1.  1.  1.  1. -1.  1. -1. -1.  1. -1.
  1. -1. -1.  1.  0.  1. -1.  1.  0.  1.  1.  1.  1.  1. -1.  1. -1.  1.
  1.  1. -1.  0.  1. -1.  0.  0. -1.  0.  1.  1.  0.  0.  1.  1.  0. -1.
 -1. -1. -1. -1.  1.  0.  1.  0.  1.  

In [90]:
im = jpeglib.read_dct(f"{stego_folder}{stego_image_path}")
print("DCT Coefficients from Stego Image:")
print(im.Y)
print(im.qt[0])

im2 = jpeglib.read_dct(f"{recovered_folder}{recovered_image}")
print("DCT Coefficients from Recovered Image:")
print(im2.Y)
print(im2.qt[0])

diff_coeff = np.array(im2.Y) - np.array(im.Y)
print("Difference Coefficients between Recovered and Stego Image:")
print(diff_coeff)

DCT Coefficients from Stego Image:
[[[[-390   82  -76 ...  -24   40    0]
   [ -22   32   78 ...  -24  -72   33]
   [   3  -33  -57 ...   55    0  -22]
   ...
   [  10  -14  -22 ...    0   23  -18]
   [ -10    0    0 ...    0    0   20]
   [   0  -18    0 ...    0  -21    0]]

  [[-390   71  -31 ...  -73  -11    0]
   [ -15  -31   -7 ...    1   13   -1]
   [ 124  -73    4 ...   10  -15    1]
   ...
   [  -6  -22  -10 ...    0   -1  -18]
   [  21   14    0 ...    0    0  -21]
   [  15   19    1 ...  -21    1    0]]

  [[-423  109   45 ...  -31   11   25]
   [  53   85   67 ...   25   -1   -1]
   [  89  -23   22 ...   34  -13  -12]
   ...
   [ -16   -6  -11 ...    0  -47  -18]
   [   9  -27    0 ...    0   25    1]
   [  29   37  -20 ...    1    0    0]]

  ...

  [[ 165   51  -21 ...  -31    1  -13]
   [  -5  111  -91 ...   25   13   -1]
   [ 127  -43  -49 ...   22  -29  -12]
   ...
   [  46    7  -45 ...  -22   -1   -1]
   [ -31  -53  -33 ...   25  -24  -20]
   [  15    0   20 ...  -39